In [ ]:
!pip install --upgrade gradio PyPDF2 deep-translator openpyxl matplotlib gTTS wordcloud textblob typer

In [ ]:
!pip install --upgrade gradio PyPDF2 deep-translator openpyxl nest-asyncio uvicorn

In [ ]:
!pip install gradio PyPDF2

In [ ]:
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/mertemin/turkish-word-list/master/words.txt", "dev_turkce_sozluk.txt")

In [ ]:
import urllib.request
# İnternetten 80.000 kelimelik dev Türkçe sözlüğü indirdilidi!
url = "https://raw.githubusercontent.com/mertemin/turkish-word-list/master/words.txt"
urllib.request.urlretrieve(url, "dev_turkce_sozluk.txt")
print("Başarılı! 'dev_turkce_sozluk.txt' dosyası indirildi.")

In [ ]:
# ==================================================================================================
# KÜTÜPHANE KURULUMU / УСТАНОВКА БИБЛИОТЕК (INSTALLATION OF DEPENDENCIES)
# TR: Google Translate API'sini kullanarak anlık makine çevirisi (Machine Translation) yapmak için gereklidir.
# RU: Требуется для выполнения мгновенного машинного перевода с использованием API Google Translate.
# ==================================================================================================
!pip install deep-translator

# -*- coding: utf-8 -*-

"""
####################################################################################################
# PROJE ADI / НАЗВАНИЕ ПРОЕКТА:
# YZ DESTEKLİ GELİŞMİŞ TÜRKÇE-RUSÇA MORFOLOJİK VE SÖZDİZİMSEL ANALİZ MOTORU
# ИИ ПРОДВИНУТЫЙ ТУРЕЦКО-РУССКИЙ МОРФОЛОГИЧЕСКИЙ И СИНТАКСИЧЕСКИЙ АНАЛИЗАТОР
#
# AKADEMİK AÇIKLAMA / АКАДЕМИЧЕСКОЕ ОПИСАНИЕ:
# TR:
# Bu proje, Doğal Dil İşleme (NLP - Natural Language Processing) teorilerine dayanan, kural tabanlı
# (Rule-Based) bir morfolojik analiz ve sözcük türü etiketleme (POS Tagging) motorudur.
# Türkçenin sondan eklemeli (agglutinative) yapısı nedeniyle oluşan milyonlarca olası kelime
# kombinasyonunu çözmek için; 'Büyük Ünlü Uyumu' (Palatal Harmony), 'Küçük Ünlü Uyumu' (Labial Harmony),
# 'Ünsüz Benzeşmesi / Sertleşmesi' (Consonant Assimilation) ve 'Ünsüz Yumuşaması' (Lenition) gibi
# karmaşık fonetik algoritmalar matematiksel olarak modellenmiştir.
#
# Sistem ayrıca sözlük tabanlı (Lexicon-based) hızlı arama ile O(1) karmaşıklığında zamir, bağlaç,
# edat ve zarf tespiti yapar. Ek olarak, Deep Learning tabanlı 'deep-translator' modülü ile
# sisteme girilen cümlenin makine çevirisini (Machine Translation) gerçekleştirir.
#
# RU:
# Этот проект представляет собой систему морфологического анализа и частеречной разметки (POS Tagging),
# основанную на правилах (Rule-Based) и теориях обработки естественного языка (NLP).
# Для решения проблемы миллионов возможных комбинаций слов, возникающих из-за агглютинативной
# природы турецкого языка, были математически смоделированы сложные фонетические алгоритмы,
# такие как 'Гармония гласных' (Palatal/Labial Harmony), 'Ассимиляция согласных' и 'Озвончение'.
#
# Система также использует словарный подход для быстрого поиска местоимений, союзов,
# послелогов и наречий со сложностью O(1). Кроме того, с помощью модуля 'deep-translator',
# основанного на глубоком обучении, выполняется машинный перевод введенного предложения.
####################################################################################################
"""

# ==================================================================================================
# GEREKLİ MODÜLLERİN İÇE AKTARILMASI / ИМПОРТ НЕОБХОДИМЫХ МОДУЛЕЙ
# ==================================================================================================
import re # TR: Düzenli ifadeler (Regex). Metni noktalama işaretlerinden arındırıp saf token'lar elde etmek için. | RU: Регулярные выражения для токенизации и очистки текста от пунктуации.
import pandas as pd # TR: Veri analizi kütüphanesi. Analiz sonuçlarını matris (Dataframe) formatında tutmak için. | RU: Библиотека для создания матриц и табличного представления данных.
from IPython.display import display # TR: Jupyter/Colab ortamında Pandas tablolarını interaktif ve grafiksel (HTML) renderlamak için. | RU: Для красивого интерактивного отображения таблиц Pandas.
from deep_translator import GoogleTranslator # TR: Derin öğrenme tabanlı Google Çeviri API'si arayüzü. | RU: Интерфейс API Google Translate на основе глубокого обучения.

# ==================================================================================================
# PANDAS GÖRSELLEŞTİRME VE MATRİS AYARLARI / НАСТРОЙКИ ВИЗУАЛИЗАЦИИ МАТРИЦ PANDAS
# TR: Uzun morfolojik analiz açıklamalarının tablo hücrelerinde kesilmesini (...) engellemek için
# pandas DataFrame'in standart karakter limitleri sonsuz (None) olarak ayarlanmıştır.
# RU: Чтобы длинные морфологические объяснения не обрезались (...) в ячейках таблицы,
# стандартные лимиты символов DataFrame установлены в бесконечность (None).
# ==================================================================================================
pd.set_option("display.max_colwidth", None) # Hücre genişliği limiti iptal. / Лимит ширины ячейки отменен.
pd.set_option("display.width", 2500)        # Ekran render genişliği 2500 piksele çıkarıldı. / Ширина рендера увеличена.
pd.set_option("display.max_rows", 500)      # Gösterilecek maksimum satır sayısı artırıldı. / Увеличено макс. кол-во строк.

# ==================================================================================================
# 1) FONETİK KURALLAR VE HARF KÜMELERİ (PHONETIC RULES & CHARACTER SETS) / ФОНЕТИЧЕСКИЕ ПРАВИЛА
# TR: Türkçedeki ek (suffix) seçim mekanizmaları tamamen ses uyumlarına dayanır.
# Bu kümeler, programın ekleri doğru seçmesi için referans aldığı çekirdek verilerdir.
# RU: Выбор суффиксов в турецком языке полностью зависит от гармонии звуков.
# Эти наборы - базовые данные для правильного подбора суффиксов программой.
# ==================================================================================================
VOWELS = "aıoueiöü"         # TR: Türkçe alfabesindeki tüm sesli (ünlü) harfler. | RU: Все гласные турецкого алфавита.
BACK_VOWELS = "aıou"        # TR: Kalın (Art damak) ünlüler. Çoğul eki (-lar/-ler) ve hâl eklerinin seçiminde kullanılır. | RU: Задние гласные (твердые). Используются для выбора суффиксов.
VOICELESS = "fstkçşhp"      # TR: Sert ünsüzler (Fıstıkçı Şahap). Bulunma (-da/-ta) ve Ayrılma (-dan/-tan) eklerindeki ünsüz benzeşmesi için. | RU: Глухие согласные. Для ассимиляции согласных.

# ==================================================================================================
# 2) SÖZCÜK TÜRÜ (POS) HASH TABLOLARI / ХЭШ-ТАБЛИЦЫ ЧАСТЕЙ РЕЧИ (POS LEXICON)
# TR: İşlemciyi yormamak ve Big-O karmaşıklığını O(1) seviyesinde tutmak için, çekim almayan
# (veya nadir alan) sözcük türleri (Zamir, Bağlaç, Edat, Ünlem, Zarf) sabit Hash Map (Dict) olarak tanımlandı.
# RU: Для снижения нагрузки на процессор и поддержания сложности на уровне O(1), неизменяемые
# части речи заданы как статические Hash Map (словари).
# ==================================================================================================
PRONOUNS = {
    "ben": ("Zamir (1. tekil)", "Местоимение (1 ед.)"), "sen": ("Zamir (2. tekil)", "Местоимение (2 ед.)"),
    "o": ("Zamir (3. tekil)", "Местоимение (3 ед.)"), "biz": ("Zamir (1. çoğul)", "Местоимение (1 мн.)"),
    "siz": ("Zamir (2. çoğul)", "Местоимение (2 мн.)"), "onlar": ("Zamir (3. çoğul)", "Местоимение (3 мн.)"),
    "bu": ("İşaret zamiri", "Указательное местоимение"), "şu": ("İşaret zamiri", "Указательное местоимение"),
}

CONJUNCTIONS = {
    "ve": ("Bağlaç", "Союз"), "ama": ("Bağlaç", "Союз"), "fakat": ("Bağlaç", "Союз"),
    "çünkü": ("Bağlaç", "Союз"), "veya": ("Bağlaç", "Союз"), "ya": ("Bağlaç", "Союз"),
}

POSTPOSITIONS = {
    "için": ("Edat (amaç)", "Послелог (для)"), "gibi": ("Edat (benzerlik)", "Послелог (как)"),
    "kadar": ("Edat", "Послелог"), "sonra": ("Edat", "Послелог"),
    "önce": ("Edat", "Послелог"), "ile": ("Edat/bağlaç", "Послелог/союз"),
}

INTERJECTIONS = {
    "merhaba": ("Ünlem/Selam", "Междометие/приветствие"), "selam": ("Ünlem/Selam", "Междометие/приветствие"),
    "hey": ("Ünlem", "Междометие"), "ah": ("Ünlem", "Междометие"), "oh": ("Ünlem", "Междометие"),
}

ADVERBS = {
    "erken": ("Zarf (zaman)", "Наречие (время)"), "geç": ("Zarf (zaman)", "Наречие (время)"),
    "bugün": ("Zarf (zaman)", "Наречие (время)"), "yarın": ("Zarf (zaman)", "Наречие (время)"),
    "dün": ("Zarf (zaman)", "Наречие (время)"), "şimdi": ("Zarf (zaman)", "Наречие (время)"),
    "hemen": ("Zarf", "Наречие"), "çok": ("Zarf (derece)", "Наречие (степени)"),
    "az": ("Zarf (derece)", "Наречие (степени)"), "sabah": ("Zarf (zaman)", "Наречие (время)"),
}

# ==================================================================================================
# 3) ÇEKİRDEK NLP VE FONETİK FONKSİYONLARI / БАЗОВЫЕ ФОНЕТИЧЕСКИЕ И NLP ФУНКЦИИ
# ==================================================================================================
def last_vowel(word: str) -> str:
    """
    TR: Kelimedeki son ünlü harfi tespit eden çekirdek algoritma. (Vowel Harmony Core)
        Türkçede eklerin hangi ünlüyle başlayacağı her zaman kökün veya bir önceki ekin son ünlüsüne bağlıdır.
        Bu fonksiyon kelimeyi tersten (reversed) tarayarak bulduğu ilk ünlü harfi döndürür.
    RU: Находит последнюю гласную в слове. Гармония гласных зависит от нее.
        Функция сканирует слово с конца (reversed) и возвращает первую найденную гласную.
    """
    for c in reversed(word):
        if c in VOWELS: return c
    return "a" # Fallback (Güvenlik mekanizması)

def soften(word: str) -> str:
    """TR: Ünsüz Yumuşaması ve İSTİSNALAR (Exceptions)."""
    # DÜZELTME: "hiç" eklendi!
    exceptions = ["hayat", "saat", "devlet", "millet", "hukuk", "sanat", "sepet", "kök", "hep", "saç", "suç", "ot", "süt", "iç", "kat", "top", "at", "hakikat", "hiç"]
    if word in exceptions:
        return word

    if word.endswith("p"): return word[:-1] + "b"
    if word.endswith("ç"): return word[:-1] + "c"
    if word.endswith("t"): return word[:-1] + "d"
    if word.endswith("k"): return word[:-1] + "ğ"
    return word

def plural(word: str) -> str:
    """TR: Arapça/Farsça kökenli bazı kelimeler kalın ünlüyle bitse de '-ler' alır."""
    # DÜZELTME: "hayat" silindi, "ihtimal" eklendi!
    exceptions_ler = ["saat", "harf", "kalp", "rol", "hal", "seyahat", "hakikat", "hayal", "dikkat", "ihtimal"]
    if word in exceptions_ler: return word + "ler"

    lv = last_vowel(word)
    return word + ("lar" if lv in BACK_VOWELS else "ler")

def tokenize(text: str) -> list:
    """
    TR: Cümleyi Token'larına ayıran fonksiyon.
        Python'un büyük 'İ' ve 'I' harflerini lower() yaparken bozmasını engellemek için ön filtre eklendi.
    """
    # Python'un meşhur i/ı lower() bug'ını manuel çözüyoruz!
    safe_text = text.replace("İ", "i").replace("I", "ı").replace("Î", "i").lower()
    return re.findall(r"[a-zçğıöşü]+", safe_text)

def suffix_from_transform(base: str, final: str) -> str:
    """
    TR: İki stringi (kök hali ve ek almış hali) kıyaslayıp, ortak prefix sonrasını (suffix) ayıklar.
        Örneğin: base="arabamız", final="arabamıza" -> suffix="a" çıktısını üretir.
    RU: Вычисляет суффикс путем сравнения начальной (base) и конечной (final) формы слова, отсекая общую часть.
    """
    i = 0
    m = min(len(base), len(final))
    while i < m and base[i] == final[i]: i += 1
    return final[i:] if i < len(final) else ""

# ==================================================================================================
# 4) MORFOLOJİK EK (SUFFIX) LİTERATÜRÜ / БАЗА ДАННЫХ И ТЕОРИЯ СУФФИКСОВ
# TR: Burada ismin alabileceği hâl (case) ve iyelik (possessive) eklerinin gramatikal açıklamaları bulunur.
# RU: Здесь находятся грамматические описания падежных и притяжательных суффиксов.
# ==================================================================================================
CASE_INFO = {
    "acc": ("Belirtme hâli (Definite Object)", "Винительный падеж", "Neyi/kimi?", "кого/что?"),
    "dat": ("Yönelme hâli (Direction)", "Дательный падеж", "Nereye/kime?", "куда/кому?"),
    "loc": ("Bulunma hâli (Location)", "Местный падеж", "Nerede?", "где?"),
    "abl": ("Ayrılma hâli (Source/Origin)", "Исходный падеж", "Nereden?", "откуда?"),
    "gen": ("Tamlayan hâli (Possessor)", "Родительный падеж", "Kimin/neyin?", "чей/чего?"),
}

POSS_INFO = {
    "ben": ("İyelik: benim", "Притяж: мой"), "sen": ("İyelik: senin", "Притяж: твой"),
    "o": ("İyelik: onun", "Притяж: его/её"), "biz": ("İyelik: bizim", "Притяж: наш"),
    "siz": ("İyelik: sizin", "Притяж: ваш"),
}

PL_INFO = ("Çoğul (Plural)", "Множ. число", "Çokluk bildirir.", "Обозначает множественность.")

# ==================================================================================================
# 5 & 6) İYELİK VE DURUM EKLERİ MATRİSİ (CASE & POSSESSIVE GENERATOR)
# ==================================================================================================
def possessives_my(word: str) -> dict:
    """TR: Kelime ünlüyle bitiyorsa iyelik eklerindeki ünlüler düşer (masa-ımız -> masamız)."""
    lv = last_vowel(word)
    w = soften(word)
    v_end = w[-1] in VOWELS

    # Ünlüyle bitiyorsa (v_end) ekin başındaki ünlüyü atarak ('ım' yerine 'm') ekle!
    if lv in "aı":
        p = ("m", "n", "sı", "mız", "nız") if v_end else ("ım", "ın", "ı", "ımız", "ınız")
    elif lv in "ei":
        p = ("m", "n", "si", "miz", "niz") if v_end else ("im", "in", "i", "imiz", "iniz")
    elif lv in "ou":
        p = ("m", "n", "su", "muz", "nuz") if v_end else ("um", "un", "u", "umuz", "unuz")
    else:
        p = ("m", "n", "sü", "müz", "nüz") if v_end else ("üm", "ün", "ü", "ümüz", "ünüz")

    return { "ben": w + p[0], "sen": w + p[1], "o": w + p[2], "biz": w + p[3], "siz": w + p[4] }

    # 3. Tekil Şahıs kuralı: Eğer kelime ünlüyle bitiyorsa zorunlu 's' kaynaştırma harfi alır (Örn: masa-sı).
    third = ("s" + p[2]) if vowel_end else p[2]
    return { "ben": w + p[0], "sen": w + p[1], "o": w + third, "biz": w + p[3], "siz": w + p[4] }

def case_forms_normal(word: str) -> dict:
    """
    TR: İsmin 5 farklı Hâl ekinin (Case Suffixes) fonetik kurallara göre uygulanması.
        Özellikle Bulunma (Locative) ve Ayrılma (Ablative) eklerinde Sert Ünsüz Benzeşmesi
        (Consonant Assimilation - d->t dönüşümü) simüle edilmiştir.
    RU: Применение 5 падежных суффиксов с учетом фонетических правил. Особое внимание уделено
        ассимиляции согласных (переход d->t) в местном и исходном падежах после глухих согласных.
    """
    lv = last_vowel(word)
    w = soften(word)
    vowel_end = w[-1] in VOWELS

    # Accusative: -ı, -i, -u, -ü. Ünlüyle bitiyorsa 'y' kaynaştırması.
    acc = w + (("y" if vowel_end else "") + ("ı" if lv in "aı" else "u" if lv in "ou" else "i" if lv in "ei" else "ü"))
    # Dative: -a, -e. Ünlüyle bitiyorsa 'y' kaynaştırması.
    dat = w + (("y" if vowel_end else "") + ("a" if lv in BACK_VOWELS else "e"))
    # Locative: Ünsüz benzeşmesi kontrolü.
    loc = word + ("ta" if word[-1] in VOICELESS and lv in BACK_VOWELS else "te" if word[-1] in VOICELESS else "da" if lv in BACK_VOWELS else "de")
    # Ablative: Ünsüz benzeşmesi kontrolü.
    abl = word + ("tan" if word[-1] in VOICELESS and lv in BACK_VOWELS else "ten" if word[-1] in VOICELESS else "dan" if lv in BACK_VOWELS else "den")
    # Genitive: -ın, -in, -un, -ün. Ünlüyle bitiyorsa 'n' kaynaştırması.
    gen = w + (("n" if vowel_end else "") + ("ın" if lv in "aı" else "un" if lv in "ou" else "in" if lv in "ei" else "ün"))

    return {"nom": word, "acc": acc, "dat": dat, "loc": loc, "abl": abl, "gen": gen}

def case_forms_after_third_possessive(word: str) -> dict:
    """
    TR: Çok Özel Gramer Kuralı: 3. Tekil Şahıs iyelik ekinden (onun) sonra hâl eki geldiğinde
        Türkçede zorunlu olarak Pronominal (Zamirsel) 'N' kaynaştırma harfi kullanılır.
        Örn: araba-sı-n-a (arabasıya DEĞİL), okul-u-n-dan.
    RU: Специальное правило грамматики: После притяжательного суффикса 3-го лица перед падежом
        обязательно вставляется прономинальная соединительная буква 'n' (напр., araba-sı-n-a).
    """
    lv = last_vowel(word)
    w = soften(word)
    vowel_end = w[-1] in VOWELS
    buffer = "n" if vowel_end else ""

    acc = w + buffer + ("ı" if lv in "aı" else "u" if lv in "ou" else "i" if lv in "ei" else "ü")
    dat = w + buffer + ("a" if lv in BACK_VOWELS else "e")
    loc = word + ("nda" if lv in BACK_VOWELS else "nde")
    abl = word + ("ndan" if lv in BACK_VOWELS else "nden")
    gen = w + buffer + ("ın" if lv in "aı" else "un" if lv in "ou" else "in" if lv in "ei" else "ün")

    return {"nom": word, "acc": acc, "dat": dat, "loc": loc, "abl": abl, "gen": gen}

# ==================================================================================================
# 8) DİNAMİK İSİM HAVUZU OLUŞTURUCU (MORPHOLOGICAL PARADIGM GENERATOR) / ГЕНЕРАТОР ПАРАДИГМ
# TR: Verilen bir 'kök' (root) kelimenin alabileceği TİPİK TÜM matematiksel ek kombinasyonlarını
#     hesaplayarak (Kök + Çoğul + İyelik + Hâl) bunları bir Hash Map (Sözlük) içinde indeksler.
# RU: Вычисляет ВСЕ типичные математические комбинации суффиксов для заданного корня
#     (Корень + Мн.ч + Притяж + Падеж) и индексирует их в Hash Map (словаре) для быстрого поиска.
# ==================================================================================================
def generate_noun_forms(root: str) -> dict:
    forms = {}
    def add(surface: str, chain: list): forms[surface] = {"root": root, "chain": chain}

    add(root, []) # Adım 0: Yalın hâl (Nominative)

    # Adım 1: Sadece Hâl (Case) ekleri
    cases = case_forms_normal(root)
    base_for_case = soften(root)

    for ck, surface in cases.items():
        if ck == "nom": continue
        base = base_for_case if ck in ("acc", "dat", "gen") else root
        suf = suffix_from_transform(base, surface)
        trn, run, trq, ruq = CASE_INFO[ck]
        add(surface, [{"surf": "-" + suf, "tr_name": trn, "ru_name": run, "tr_why": trq, "ru_why": ruq}])

    # Adım 2: İyelik (Possessive) ve İyelik + Hâl kombinasyonları
    poss = possessives_my(root)
    for person, poss_surface in poss.items():
        base_poss = soften(root)
        suf_poss = suffix_from_transform(base_poss, poss_surface)
        trp, rup = POSS_INFO[person]
        add(poss_surface, [{"surf": "-" + suf_poss, "tr_name": trp, "ru_name": rup, "tr_why": "Aidiyet bildirir.", "ru_why": "Обозначает принадлежность."}])

        # 3. Şahıs (O) için özel kaynaştırma (Pronominal N) fonksiyonu çağrılır
        cf = case_forms_after_third_possessive(poss_surface) if person == "o" else case_forms_normal(poss_surface)
        base_surface = soften(poss_surface)

        for ck, surface2 in cf.items():
            if ck == "nom": continue
            base = base_surface if ck in ("acc", "dat", "gen") else poss_surface
            suf2 = suffix_from_transform(base, surface2)
            trn, run, trq, ruq = CASE_INFO[ck]
            add(surface2, [
                {"surf": "-" + suf_poss, "tr_name": trp, "ru_name": rup, "tr_why": "Aidiyet bildirir.", "ru_why": "Обозначает принадлежность."},
                {"surf": "-" + suf2, "tr_name": trn, "ru_name": run, "tr_why": trq, "ru_why": ruq}
            ])

    # Adım 3: Çoğul Eki (-lar/-ler) Kombinasyonları
    pl = plural(root)
    pl_suf = suffix_from_transform(root, pl)
    add(pl, [{"surf": "-" + pl_suf, "tr_name": PL_INFO[0], "ru_name": PL_INFO[1], "tr_why": PL_INFO[2], "ru_why": PL_INFO[3]}])

    # Çoğul + Hâl Ekleri
    pl_cases = case_forms_normal(pl)
    base_pl = soften(pl)

    for ck, surface in pl_cases.items():
        if ck == "nom": continue
        base = base_pl if ck in ("acc", "dat", "gen") else pl
        suf = suffix_from_transform(base, surface)
        trn, run, trq, ruq = CASE_INFO[ck]
        add(surface, [
            {"surf": "-" + pl_suf, "tr_name": PL_INFO[0], "ru_name": PL_INFO[1], "tr_why": PL_INFO[2], "ru_why": PL_INFO[3]},
            {"surf": "-" + suf, "tr_name": trn, "ru_name": run, "tr_why": trq, "ru_why": ruq}
        ])

    # En Uzun Zincir: Çoğul + İyelik + Hâl Ekleri
    pl_poss = possessives_my(pl)
    for person, poss_surface in pl_poss.items():
        base_poss = soften(pl)
        suf_poss = suffix_from_transform(base_poss, poss_surface)
        trp, rup = POSS_INFO[person]
        add(poss_surface, [
            {"surf": "-" + pl_suf, "tr_name": PL_INFO[0], "ru_name": PL_INFO[1], "tr_why": PL_INFO[2], "ru_why": PL_INFO[3]},
            {"surf": "-" + suf_poss, "tr_name": trp, "ru_name": rup, "tr_why": "Aidiyet bildirir.", "ru_why": "Обозначает принадлежность."}
        ])

        cf = case_forms_after_third_possessive(poss_surface) if person == "o" else case_forms_normal(poss_surface)
        base_surface = soften(poss_surface)
        for ck, surface2 in cf.items():
            if ck == "nom": continue
            base = base_surface if ck in ("acc", "dat", "gen") else poss_surface
            suf2 = suffix_from_transform(base, surface2)
            trn, run, trq, ruq = CASE_INFO[ck]
            add(surface2, [
                {"surf": "-" + pl_suf, "tr_name": PL_INFO[0], "ru_name": PL_INFO[1], "tr_why": PL_INFO[2], "ru_why": PL_INFO[3]},
                {"surf": "-" + suf_poss, "tr_name": trp, "ru_name": rup, "tr_why": "Aidiyet bildirir.", "ru_why": "Обозначает принадлежность."},
                {"surf": "-" + suf2, "tr_name": trn, "ru_name": run, "tr_why": trq, "ru_why": ruq}
            ])

    return forms

# ==================================================================================================
# 9) FİİL VE ARAÇ EKİ (INSTRUMENTAL) TANIMA SİSTEMİ / АНАЛИЗ ГЛАГОЛОВ И ОРУДИЙНОГО ПАДЕЖА
# ==================================================================================================
def looks_like_verb(token: str) -> bool:
    endings = [
        "yorum","yorsun","yor","yoruz","yorsunuz","yorlar",
        "dım","dim","dum","düm","tım","tim","tum","tüm","dık","dik","duk","dük","tık","tik","tuk","tük","dı","di","du","dü","tı","ti","tu","tü",
        "acağım","eceğim","acaksın","eceksin","acak","ecek","acağız","eceğiz","acaksınız","eceksiniz",
        "mış","miş","muş","müş", "mıştı", "mişti", "muştu", "müştü",
        "mak","mek","ıp","ip","up","üp","arak","erek", "ıncaya", "inceye", "uncaya", "ünceye",
        "maksızın", "meksizin", "ken", "yorken", "iyorken",
        "eceğimiz", "acağımız", "eceğiniz", "acağınız", "eceğini", "acağını", "eceğinizi", "acağınızı", "eceği", "acağı", "eceklerine", "acaklarına",
        "diğimiz", "dığımız", "duğumuz", "düğümüz", "tiğimiz", "tığımız", "tuğumuz", "tüğümüz",
        "diğiniz", "dığınız", "duğunuz", "düğünüz", "tiğiniz", "tığınız", "tuğunuz", "tüğünüz",
        "diğimizi", "dığımızı", "duğumuzu", "düğümüzü", "tiğimizi", "tığımızı", "tuğunuzu", "tüğünüzü",
        "dığımızda","diğimizde","tığımızda","tiğimizde", "dığınızda", "diğinizde", "duğunuzda", "düğünüzde", "tığınızda", "tiğinizde", "tuğunuzda", "tüğünüzde",
        "dığı","diği","duğu","düğü","tığı","tiği","tuğu","tüğü", "dığını", "diğini", "duğunu", "düğünü", "tığını", "tiğini", "tuğunu", "tüğünü",
        "dıkları", "dikleri", "dukları", "dükleri", "tıkları", "tikleri", "tukları", "tükleri", "dıklarımızdan", "diklerimizden",
        "mesinin", "masını", "memizden", "mamızdan", "mesi", "ması", "meyen", "mayan", "en", "an", "me", "ma", "mekte", "makta",
        "melerinin", "malarının", "melerini", "malarını", "meleri", "maları", "maz", "mez"
    ]
    return any(token.lower().endswith(e) for e in endings)

def analyze_verb(token: str) -> dict:
    t = token.lower()
    for suf in ["yorum","yorsun","yor","yoruz","yorsunuz","yorlar"]:
        if t.endswith(suf): return {"root": t[:-len(suf)] or "-", "tr": f"Fiil: Şimdiki zaman + kişi: {suf}.", "ru": f"Глагол: наст. время + лицо: {suf}."}
    for suf in ["dım","dim","dum","düm","tım","tim","tum","tüm","dık","dik","duk","dük","tık","tik","tuk","tük","dı","di","du","dü","tı","ti","tu","tü"]:
        if t.endswith(suf): return {"root": t[:-len(suf)] or "-", "tr": f"Fiil: Geçmiş zaman + kişi: {suf}.", "ru": f"Глагол: прош. время + лицо: {suf}."}
    for suf in ["mış","miş","muş","müş", "mıştı", "mişti"]:
        if t.endswith(suf): return {"root": t[:-len(suf)] or "-", "tr": "Fiil: Duyulan Geçmiş Zaman (-mış/-miş).", "ru": "Глагол: неочевидное прошедшее."}
    for suf in ["acağım","eceğim","acaksın","eceksin","acak","ecek","acağız","eceğiz","acaksınız","eceksiniz"]:
        if t.endswith(suf): return {"root": t[:-len(suf)] or "-", "tr": f"Fiil: Gelecek zaman + kişi: {suf}.", "ru": f"Глагол: буд. время + лицо: {suf}."}

    for suf in ["eceğimiz", "acağımız", "eceğiniz", "acağınız", "eceğini", "acağını", "eceğinizi", "acağınızı", "eceği", "acağı", "eceklerine", "acaklarına", "diğimiz", "dığımız", "duğumuz", "düğümüz", "tiğimiz", "tığımız", "tuğumuz", "tüğümüz", "diğiniz", "dığınız", "duğunuz", "düğünüz", "tiğiniz", "tığınız", "tuğunuz", "tüğünüz", "diğimizi", "dığımızı", "duğumuzu", "düğümüzü", "tiğimizi", "tığımızı", "tuğunuzu", "tüğünüzü", "dığımızda","diğimizde","tığımızda","tiğimizde", "dığınızda", "diğinizde", "duğunuzda", "düğünüzde", "tığınızda", "tiğinizde", "tuğunuzda", "tüğünüzde", "dığı","diği","duğu","düğü","tığı","tiği","tuğu","tüğü", "dığını", "diğini", "duğunu", "düğünü", "tığını", "tiğini", "tuğunu", "tüğünü", "dıkları", "dikleri", "dukları", "dükleri", "tıkları", "tikleri", "tukları", "tükleri", "dıklarımızdan", "diklerimizden", "mesinin", "masını", "memizden", "mamızdan", "mesi", "ması", "meyen", "mayan", "melerinin", "malarının", "melerini", "malarını", "meleri", "maları", "maz", "mez"]:
        if t.endswith(suf): return {"root": t[:-len(suf)] or "-", "tr": "Fiil: Fiilimsi (Sıfat/İsim-Fiil) + İyelik/Hâl.", "ru": "Глагол: причастие/отглагольное имя + аффиксы."}

    if t.endswith("mak") or t.endswith("mek"): return {"root": t[:-3] or "-", "tr": "Fiil: Mastar (Infinitive).", "ru": "Глагол: инфинитив."}
    if t.endswith(("ıp","ip","up","üp")): return {"root": t[:-2] or "-", "tr": "Fiil: Ulaç / Zarf-Fiil (-ip).", "ru": "Глагол: деепричастие."}
    if t.endswith(("arak","erek")): return {"root": t[:-4] or "-", "tr": "Fiil: Ulaç / Zarf-Fiil (-arak/-erek).", "ru": "Глагол: деепричастие."}
    if t.endswith(("ıncaya", "inceye", "uncaya", "ünceye")): return {"root": t[:-6] or "-", "tr": "Fiil: Ulaç / Zarf-Fiil (-ıncaya).", "ru": "Глагол: деепричастие."}
    if t.endswith(("maksızın","meksizin")): return {"root": t[:-8] or "-", "tr": "Fiil: Ulaç (-maksızın).", "ru": "Глагол: деепричастие."}
    if t.endswith(("ken", "yorken", "iyorken")): return {"root": t.replace("iyorken","").replace("yorken","").replace("ken","") or "-", "tr": "Fiil: Ulaç (-ken).", "ru": "Глагол: деепричастие."}
    if t.endswith(("en", "an")): return {"root": t[:-2] or "-", "tr": "Fiil: Sıfat-Fiil (-an/-en).", "ru": "Глагол: причастие."}
    if t.endswith(("mekte", "makta")): return {"root": t[:-5] or "-", "tr": "Fiil: Şimdiki Zaman (Resmi) (-makta/-mekte).", "ru": "Глагол: наст. время."}
    if t.endswith(("me", "ma")) and len(t) > 4: return {"root": t[:-2] or "-", "tr": "Fiil: İsim-Fiil (-ma/-me).", "ru": "Глагол: отглагольное имя."}

    return {"root": "-", "tr": "Fiil: (Kapsam dışı karmaşık çekim).", "ru": "Глагол: (сложное спряжение)."}

# 🚀 İŞTE YANLIŞLIKLA SİLİNEN KÜÇÜK FONKSİYON BURADA!
def looks_like_instrumental(token: str) -> bool:
    return token.endswith(("le","la","lerle","larla","yle","yla"))

# 🚀 İŞTE YANLIŞLIKLA SİLİNEN KÜÇÜK FONKSİYON BURADA!
def looks_like_instrumental(token: str) -> bool:
    return token.endswith(("le","la","lerle","larla","yle","yla"))
# ==================================================================================================
# 11) GÖRSEL ÇIKTI DÜZENLEYİCİLERİ (STRING FORMATTING) / ФОРМАТИРОВАНИЕ ВЫВОДА
# TR: Tablolardaki alt satıra geçme (\n) bozulmalarını engellemek için çift dik çizgi (||) kullanıldı.
# RU: Использование ' || ' вместо '\n' для предотвращения искажения таблиц при рендеринге в браузере.
# ==================================================================================================
def chain_short_tr(root: str, chain: list) -> str:
    if not chain: return f"{root} (yalın)"
    return " + ".join([root] + [c["surf"].replace("-", "") for c in chain])

def chain_short_ru(root: str, chain: list) -> str:
    if not chain: return f"{root} (именит.)"
    return " + ".join([root] + [c["surf"].replace("-", "") for c in chain])

def chain_long_tr(chain: list) -> str:
    if not chain: return "Ek yok (yalın biçim / nominative)."
    return "  ||  ".join([f"{i}) {c['surf']} → {c['tr_name']} | {c['tr_why']}" for i, c in enumerate(chain, start=1)])

def chain_long_ru(chain: list) -> str:
    if not chain: return "Нет суффиксов (именительный падеж)."
    return "  ||  ".join([f"{i}) {c['surf']} → {c['ru_name']} | {c['ru_why']}" for i, c in enumerate(chain, start=1)])

# ==================================================================================================
# 12) VERİ TABANI YÜKLEMESİ VE SÖZLÜK (ROOT DICTIONARY) / БАЗА КОРНЕЙ
# TR: Sistem dışarıdan bir dosya bulamazsa, hata (crash) vermemesi için bellek içi (In-Memory) kök havuzu.
# RU: In-Memory база корней для предотвращения сбоев, если внешний файл не загружен.
# ==================================================================================================
DEFAULT_ROOTS = [
    "okul","ev","kitap","araba","yol","kalem","masa","şehir","arkadaş","kapı","su","çocuk","gün","ders",
    "üniversite","anne","baba","telefon","market","kütüphane","otobüs","bilet","bahçe","öğretmen","sınıf","oda", "ödev",

    # 🚀 YENİ EKLENEN YEDEK TEST KELİMELERİ (Dosya silinse bile programın çökmesini engeller)
    "dikkat", "saat", "beyin", "fırtına", "netice", "kalp", "umut", "zihin", "ihtimal", "art", "temel", "mantık",
    "dünya", "hiçbir", "hukuk", "devlet", "güç", "evrensel", "bir", "özgürlük", "arayış", "araştırmacı", "yıl",
    "üzeri", "devasa", "proje", "bilgisayar", "anakart", "test", "problem", "yepyeni", "algoritma", "dize", "inanç",
    "toplum", "iç", "suç", "hata", "yüz", "hiç", "kıymet", "tarih", "sahne", "üst", "kritik", "rol", "laboratuvar",
    "yapay", "zeka", "sistem", "kelime", "kök", "gizli", "anlam", "fonetik", "kural", "dilbilgisel", "istisna",
    "aşılmaz", "engel", "teker", "insanlık", "gelecek", "yegane", "hakikat", "veri", "taktir", "şayan", "doğa", "dil",
    "işlem", "uzman", "akılalmaz", "morfolojik", "yapı", "sonsuz", "ek", "zincir", "ömür", "büyük", "kısım", "kod",
    "hayal", "seviye", "program", "karmaşıklık", "cümle", "tarifsiz", "heyecan", "muhteşem", "yazılım", "sadece",
    "ufacık", "başlangıç", "zorluk", "boyun", "sınır", "paramparça", "vazgeç"
]

def load_roots(path: str) -> list:
    roots = []
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if w := line.strip().lower(): roots.append(w)
    except Exception: return [] # Hata toleransı (Fault tolerance)
    return roots
# ==================================================================================================
# 13 & 14) ANA ANALİZ MOTORU VE DATAFRAME İNŞASI (PIPELINE ENGINE) / ГЛАВНЫЙ АНАЛИЗАТОР
# ==================================================================================================
def analyze_token(token: str, roots: list) -> dict:
    t = token.lower()

    # 🚀 YENİ YAMA: -ki, -dır, -dırlar ve -çe eklerini EN BAŞTA temizle!
    clean_t = t
    if clean_t.endswith("ki") and len(clean_t) > 4: clean_t = clean_t[:-2]
    if clean_t.endswith(("dırlar","dirler","durlar","dürler","tırlar","tirler","turlar","türler")) and len(clean_t) > 8: clean_t = clean_t[:-6]
    elif clean_t.endswith(("dır","dir","dur","dür","tır","tir","tur","tür")) and len(clean_t) > 5: clean_t = clean_t[:-3]
    if clean_t.endswith(("çe", "ça")) and len(clean_t) > 4: clean_t = clean_t[:-2]

    if clean_t in PRONOUNS: return {"word": t, "pos_tr": PRONOUNS[clean_t][0], "pos_ru": PRONOUNS[clean_t][1], "root": clean_t, "tr": "Zamir (Kişi/İşaret)", "ru": "Местоимение", "chain": None}
    if clean_t in CONJUNCTIONS: return {"word": t, "pos_tr": CONJUNCTIONS[clean_t][0], "pos_ru": CONJUNCTIONS[clean_t][1], "root": clean_t, "tr": "Bağlaç (Conjunction)", "ru": "Союз", "chain": None}
    if clean_t in POSTPOSITIONS: return {"word": t, "pos_tr": POSTPOSITIONS[clean_t][0], "pos_ru": POSTPOSITIONS[clean_t][1], "root": clean_t, "tr": "Edat (Postposition)", "ru": "Послелог", "chain": None}
    if clean_t in INTERJECTIONS: return {"word": t, "pos_tr": INTERJECTIONS[clean_t][0], "pos_ru": INTERJECTIONS[clean_t][1], "root": clean_t, "tr": "Ünlem (Interjection)", "ru": "Междометие", "chain": None}
    if clean_t in ADVERBS: return {"word": t, "pos_tr": ADVERBS[clean_t][0], "pos_ru": ADVERBS[clean_t][1], "root": clean_t, "tr": "Zarf (Adverb)", "ru": "Наречие", "chain": None}

    if looks_like_verb(clean_t):
        v = analyze_verb(clean_t)
        return {"word": t, "pos_tr": "Fiil", "pos_ru": "Глагол", "root": v["root"], "tr": v["tr"], "ru": v["ru"], "chain": None}

    if looks_like_instrumental(clean_t): return {"word": t, "pos_tr": "İsim (araç eki)", "pos_ru": "Имя (-le/-la)", "root": "-", "tr": "Araç/vasıta anlamı taşır.", "ru": "Орудие/средство.", "chain": None}

    # 4. Aşama: AĞIR MORFOLOJİK ÇÖZÜMLEME
    best = None
    for r in roots:
        if len(r) > len(clean_t): continue
        if not clean_t.startswith(r) and not clean_t.startswith(soften(r)): continue

        forms = generate_noun_forms(r)
        if clean_t in forms:
            cand = forms[clean_t]
            if best is None or len(cand["root"]) > len(best["root"]): best = cand

    if best:
        return {"word": t, "pos_tr": "İsim (Noun)", "pos_ru": "Существительное", "root": best["root"], "tr": "Kök ve morfolojik ek zinciri başarıyla haritalandı.", "ru": "Найдена цепочка суффиксов.", "chain": best["chain"]}

    return {"word": t, "pos_tr": "Bilinmiyor (Out of Vocabulary)", "pos_ru": "Неизвестно", "root": "-", "tr": "Sistem sözlüğünde kök bulunamadı.", "ru": "Корень не найден в словаре.", "chain": None}

# 🚀 İŞTE YANLIŞLIKLA SİLİNEN O EFSANE FONKSİYON BURADA!
def analyze_sentence(text: str, roots: list):
    """Cümleyi tokenlara böler, analiz eder ve Pandas DataFrame (Tablo) üretir."""
    tokens = tokenize(text)
    pos_data = []
    morph_data = []

    for i, token in enumerate(tokens):
        res = analyze_token(token, roots)

        # 1. Tablo Verileri (Sözcük Türü)
        pos_data.append({
            "Sıra / №": i, "Kelime / Слово": token, "Tür (TR)": res["pos_tr"],
            "Тип (RU)": res["pos_ru"], "Kök / Корень": res["root"],
            "Açıklama (TR)": res["tr"], "Пояснение (RU)": res["ru"]
        })

        # 2. Tablo Verileri (Morfoloji Zinciri)
        if res["chain"] is not None:
            morph_data.append({
                "Sıra / №": i, "Kelime / Слово": token, "Kök / Корень": res["root"],
                "Kısa zincir (TR)": chain_short_tr(res["root"], res["chain"]),
                "Коротко (RU)": chain_short_ru(res["root"], res["chain"]),
                "Uzun açıklama (TR)": chain_long_tr(res["chain"]),
                "Длинно (RU)": chain_long_ru(res["chain"])
            })
        else:
            morph_data.append({
                "Sıra / №": i, "Kelime / Слово": token, "Kök / Корень": res["root"],
                "Kısa zincir (TR)": f"{token} (yalın)", "Коротко (RU)": f"{token} (именит.)",
                "Uzun açıklama (TR)": "Ek yok (yalın biçim / nominative).",
                "Длинно (RU)": "Нет суффиксов (именительный падеж)."
            })

    return pd.DataFrame(pos_data), pd.DataFrame(morph_data)
# ==================================================================================================
# 15) WEB ARAYÜZÜ V4 - BİLİNGUAL (TÜRKÇE/RUSÇA) ARAYÜZ VE HATA KORUMALI MOTOR
# ==================================================================================================
import os
import re
import gradio as gr
import PyPDF2
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from deep_translator import GoogleTranslator
from openpyxl.styles import Alignment
from gtts import gTTS
from wordcloud import WordCloud
from collections import Counter

gr.close_all()

# RUSÇA POS HARİTASI (Grafik Etiketleri İçin)
POS_RU_MAP = {
    "İsim (Noun)": "Сущ.",
    "Fiil (Verb)": "Глагол",
    "Sıfat (Adj)": "Прил.",
    "Zamir (Pron)": "Мест.",
    "Zarf (Adv)": "Нареч.",
    "Edat (Postp)": "Предл.",
    "Bağlaç (Conj)": "Союз",
    "Bilinmiyor": "Неизв."
}

# SÖZLÜK AYARLARI
if os.path.exists("dev_turkce_sozluk.txt"):
    roots = load_roots("dev_turkce_sozluk.txt")
else:
    roots = DEFAULT_ROOTS

roots.extend(["insanoğlu", "yegane", "birbir", "hakikat", "yüzyıl", "zeka", "dilbilgisel", "müddet", "zorun", "asl", "zihn", "ömr", "kusursuzca", "satırlık"])

def extract_text_from_pdf(pdf_file_path):
    text = ""
    try:
        with open(pdf_file_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                if page.extract_text():
                    text += page.extract_text() + " "
    except Exception as e:
        return f"PDF Okuma Hatası / Ошибка чтения PDF: {e}"
    return text

def make_html_table(df):
    html = df.to_html(index=False)
    html = html.replace('<table border="1" class="dataframe">', '<table style="width:100%; border-collapse: collapse; font-family: Arial; font-size: 14px; text-align: left; background-color: #ffffff !important; color: #000000 !important;" border="1">')
    html = html.replace('<th>', '<th style="background-color: #2D3748 !important; color: #ffffff !important; padding: 12px; border: 1px solid #cccccc !important; font-weight: bold;">')
    html = html.replace('<td>', '<td style="padding: 10px; border: 1px solid #cccccc !important; color: #000000 !important; background-color: #ffffff !important;">')
    return html

def get_keywords(text):
    words = [w.lower() for w in re.findall(r'\w+', text) if len(w) > 3]
    top_5 = [w[0] for w in Counter(words).most_common(5)]
    if not top_5: return "Yeterli kelime yok. / Недостаточно слов."
    return " ".join([f"<span style='background-color:#E2E8F0; color:#2D3748; padding:5px 10px; border-radius:15px; font-weight:bold; margin-right:5px;'>#{w}</span>" for w in top_5])

# --- ANA MOTOR ÇALIŞTIRICISI ---
def process_nlp(pdf_file, text_input):
    final_text = ""
    if pdf_file is not None:
        final_text = extract_text_from_pdf(pdf_file.name)
    elif text_input.strip() != "":
        final_text = text_input
    else:
        return "⚠️ Lütfen bir PDF yükleyin veya metin girin. / Пожалуйста, загрузите PDF или введите текст.", None, "", "", "", "", None, None, None

    # 1. Çeviri ve Ses (TTS)
    try:
        ceviri = GoogleTranslator(source='tr', target='ru').translate(final_text)
        tts = gTTS(text=ceviri, lang='ru')
        audio_path = "ru_ses.mp3"
        tts.save(audio_path)
    except:
        ceviri = "❌ Çeviri Hatası. / Ошибка перевода."
        audio_path = None

    keywords_html = get_keywords(final_text)

    # 2. Morfolojik Çözümleme Tabloları
    df_pos, df_morph = analyze_sentence(final_text, roots)
    html_pos = make_html_table(df_pos)
    html_morph = make_html_table(df_morph)

    # 3. En Çok Kullanılan Ekler (Suffix Frequency) - HATA KORUMALI
    suffix_col = next((col for col in df_morph.columns if "Ek" in col or "Açıklama" in col or "Suffix" in col), None)

    if suffix_col:
        suffixes = [str(row[suffix_col]) for _, row in df_morph.iterrows() if str(row[suffix_col]).strip() not in ['—', '', 'nan', 'None']]
        suffix_counts = Counter(suffixes)

        if suffix_counts:
            df_suffix_summary = pd.DataFrame(suffix_counts.items(), columns=['Gramatikal Yapı / Грамматическая структура', 'Frekans / Частота']).sort_values(by='Frekans / Частота', ascending=False)
            html_suffix_summary = make_html_table(df_suffix_summary.head(10))
        else:
            html_suffix_summary = "<p style='color:black;'>Çözümlenebilir ek bulunamadı. / Суффиксы не найдены.</p>"
    else:
        html_suffix_summary = f"<p style='color:black;'>Tabloda uygun sütun bulunamadı. / Подходящий столбец не найден. Mevcut: {', '.join(df_morph.columns)}</p>"

    # 4. BİLİNGUAL GRAFİK
    fig_bar, ax_bar = plt.subplots(figsize=(10, 6))
    fig_bar.patch.set_facecolor('white')
    ax_bar.set_facecolor('white')
    try:
        counts = df_pos.iloc[:, 2].value_counts()
        labels = [f"{label}\n({POS_RU_MAP.get(label, label)})" for label in counts.index]

        bars = ax_bar.bar(labels, counts.values, color=['#4299e1', '#48bb78', '#ed8936', '#ecc94b', '#f56565', '#9f7aea'])
        ax_bar.set_title("Sözcük Türü Dağılımı / Распределение частей речи", fontsize=14, fontweight='bold', color='black')
        ax_bar.set_ylabel("Frekans / Количество", color='black', fontsize=12)
        ax_bar.tick_params(colors='black')

        plt.xticks(rotation=45, ha='right', fontsize=10, color='black')
        fig_bar.tight_layout()
    except:
        ax_bar.text(0.5, 0.5, "Görselleştirmek için yeterli veri yok. / Недостаточно данных.", ha='center', color='black')

    # 5. Kelime Bulutu (Word Cloud)
    fig_wc, ax_wc = plt.subplots(figsize=(8, 4))
    try:
        wc = WordCloud(width=800, height=400, background_color="white", colormap="viridis").generate(final_text)
        ax_wc.imshow(wc, interpolation='bilinear')
        ax_wc.axis("off")
        fig_wc.tight_layout()
    except:
        ax_wc.text(0.5, 0.5, "Kelime bulutu oluşturulamadı. / Не удалось создать облако слов.", ha='center')

    # 6. Excel Çıktısı
    report_path = "NLP_Analiz_Raporu.xlsx"
    try:
        with pd.ExcelWriter(report_path, engine='openpyxl') as writer:
            df_pos.to_excel(writer, sheet_name="POS_Tagging", index=False)
            df_morph.to_excel(writer, sheet_name="Morphology", index=False)
            for sheet in writer.sheets.values():
                for col in sheet.columns:
                    for cell in col:
                        cell.alignment = Alignment(wrap_text=True, vertical="top")
                    sheet.column_dimensions[col[0].column_letter].width = 45
    except Exception:
        report_path = None

    return ceviri, audio_path, keywords_html, html_pos, html_morph, html_suffix_summary, fig_bar, fig_wc, report_path

# --- SEKME TASARIMLI (DASHBOARD) ARAYÜZ ---
custom_css = """
body, * { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif !important; }
body { background-color: #f0f4f8; }
.header-box { text-align: center; background-color: #ffffff; border-radius: 8px; border: 1px solid #e2e8f0; padding: 20px; margin-bottom: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); }
.gr-box { background-color: #ffffff; border-radius: 8px; border: 1px solid #e2e8f0; padding: 20px; margin-bottom: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); }
.plot-container { text-align: center; background-color: #ffffff; padding: 15px; border-radius: 8px; margin-top: 15px;}
#analyze_btn { background: linear-gradient(135deg, #4299e1 0%, #3182ce 100%) !important; border: none !important; color: white !important; font-weight: bold; font-size: 16px; border-radius: 8px; margin-top: 10px; margin-bottom: 15px; }
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as interface:
    with gr.Column(elem_classes="header-box"):
        gr.Markdown("<h1 style='color: #2D3748; margin-bottom: 0;'>🤖 Profesyonel NLP & Veri Bilimi Paneli</h1><h3 style='color: #4A5568; margin-top: 5px;'>Продвинутая панель NLP и науки о данных</h3>")

    with gr.Row(elem_classes="gr-box"):
        with gr.Column(scale=1):
            pdf_input = gr.File(label="📄 1. PDF Yükle / Загрузить PDF", file_types=[".pdf"])
        with gr.Column(scale=2):
            text_input = gr.Textbox(label="✍️ 2. Veya Metin Girin / Или введите текст", lines=5)

    analyze_btn = gr.Button("🚀 MOTORU ATEŞLE / ЗАПУСТИТЬ АНАЛИЗ", variant="primary", elem_id="analyze_btn")

    with gr.Tabs():
        # SEKME 1
        with gr.TabItem("🎙️ 1) Çeviri & Ses / Перевод и Звук"):
            with gr.Column(elem_classes="gr-box"):
                gr.Markdown("### 🇷🇺 Rusça Çeviri ve Sesli Asistan / Русский перевод и голосовой помощник")
                translation_output = gr.Textbox(label="Rusça Metin / Русский текст", lines=3, interactive=False)
                audio_output = gr.Audio(label="Dinle / Слушать", type="filepath")
                gr.Markdown("<hr>### 🔑 Metnin Anahtar Kelimeleri / Ключевые слова")
                keywords_output = gr.HTML()

        # SEKME 2
        with gr.TabItem("📋 2) Morfoloji / Морфология"):
            with gr.Column(elem_classes="gr-box"):
                gr.Markdown("### 📊 En Sık Kullanılan Morfolojik Ekler / Часто используемые суффиксы")
                gr.Markdown("*Bu tablo, metnin gramatikal yapısının röntgenini çeker. / Эта таблица показывает грамматическую структуру текста.*")
                suffix_output = gr.HTML()
                gr.Markdown("<hr>### 📋 Sözcük Türü Tablosu / Таблица частей речи (POS)")
                pos_output = gr.HTML()
                gr.Markdown("<hr>### 🧬 Detaylı Morfolojik Çözümleme / Морфологический анализ")
                morph_output = gr.HTML()
                excel_output = gr.File(label="📥 Excel Olarak İndir / Скачать отчет в Excel")

        # SEKME 3
        with gr.TabItem("📈 3) Görselleştirme / Визуализация"):
            with gr.Column(elem_classes="plot-container"):
                gr.Markdown("### 📊 Sözcüklerin Türlerine Göre Dağılımı / Распределение частей речи")
                bar_plot_output = gr.Plot()
                gr.Markdown("<hr>### ☁️ Kelime Bulutu / Облако слов")
                wordcloud_output = gr.Plot()

    analyze_btn.click(
        fn=process_nlp,
        inputs=[pdf_input, text_input],
        outputs=[
            translation_output, audio_output, keywords_output,
            pos_output, morph_output, suffix_output, bar_plot_output, wordcloud_output, excel_output
        ]
    )

print("⏳ TR: Sistem Başlatılıyor... / RU: Система запускается...")
interface.launch(share=True, debug=True)

/tmp/ipykernel_16218/340677827.py:692: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as interface:


⏳ TR: Sistem Başlatılıyor... / RU: Система запускается...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2cc80b5c0e1a80764f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
